# Load Gold Datasets

In [0]:
# Load all 5 Gold layer aggregation tables
# These tables are ready for export to CSV and tables
base_path = "/Volumes/weather_catalog/weather_platform/weather_data"

gold_path = f"{base_path}/gold"

yearly_df = spark.read.parquet(f"{gold_path}/yearly_city_summary")
monthly_df = spark.read.parquet(f"{gold_path}/monthly_city_summary")
rainfall_df = spark.read.parquet(f"{gold_path}/rainfall_ranking")
trend_df = spark.read.parquet(f"{gold_path}/temperature_trend")
extremes_df = spark.read.parquet(f"{gold_path}/weather_extremes")

print("Gold datasets loaded successfully")

# Create Export Folder

In [0]:
# Define export path for CSV files (for external consumption or download)
export_path = "/Volumes/weather_catalog/weather_platform/weather_data/export"

print(export_path)

# Export Yearly Summary

In [0]:
# Export yearly_city_summary to single CSV file with headers
# coalesce(1) ensures single output file instead of multiple partitions
yearly_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    f"{export_path}/yearly_city_summary"
)
display(dbutils.fs.ls(f"{export_path}/yearly_city_summary"))

# Export Monthly Summary

In [0]:
# Export monthly_city_summary to single CSV file with headers
monthly_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    f"{export_path}/monthly_city_summary"
)
display(dbutils.fs.ls(f"{export_path}/monthly_city_summary"))

# Export Remaining Datasets

## Rainfall Ranking

In [0]:
# Export rainfall_ranking to single CSV file with headers
# Contains cities ranked by rainfall within each year
rainfall_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    f"{export_path}/rainfall_ranking"
)

## Temperature Trend

In [0]:
# Export temperature_trend to single CSV file with headers
# Contains year-over-year temperature changes per city
trend_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    f"{export_path}/temperature_trend"
)

## Weather Extremes

In [0]:
# Export weather_extremes to single CSV file with headers
# Contains highest/lowest temperatures and max daily rainfall per city per year
extremes_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    f"{export_path}/weather_extremes"
)

# Verify Export Folder

In [0]:
# Verify all 5 export folders were created successfully
display(dbutils.fs.ls(export_path))

# Rename folder names

In [0]:
# List all CSV files inside each export folder
# Each folder contains: _SUCCESS marker, CSV file, and metadata files
folders = [
    "yearly_city_summary",
    "monthly_city_summary",
    "rainfall_ranking",
    "temperature_trend",
    "weather_extremes"
]

for folder in folders:
    print(f"\n{folder}")
    display(dbutils.fs.ls(f"{export_path}/{folder}"))

# Check Whether tables exists

In [0]:
# Check if tables already exist in weather_platform schema
# Used to verify table creation before attempting to create them
spark.sql("""
SHOW TABLES IN weather_catalog.weather_platform
""").show(truncate=False)

# Check the error occured

In [0]:
# Debug: List all Gold layer folders to verify parquet files exist
# Should show 5 folders: yearly_city_summary, monthly_city_summary, rainfall_ranking, temperature_trend, weather_extremes
display(dbutils.fs.ls("/Volumes/weather_catalog/weather_platform/weather_data/gold"))

In [0]:
# Debug: Verify yearly_city_summary parquet files exist and are readable
display(dbutils.fs.ls("/Volumes/weather_catalog/weather_platform/weather_data/gold/yearly_city_summary"))

# Saving as tables for Streamlit

In [0]:
# Create tables from Gold layer parquet files
# These tables enable SQL queries and Streamlit app integration
# Tables: yearly_city_summary, monthly_city_summary, rainfall_ranking, temperature_trend, weather_extremes
base_path = "/Volumes/weather_catalog/weather_platform/weather_data"
gold_path = f"{base_path}/gold"

yearly_df = spark.read.parquet(f"{gold_path}/yearly_city_summary")
monthly_df = spark.read.parquet(f"{gold_path}/monthly_city_summary")
rainfall_df = spark.read.parquet(f"{gold_path}/rainfall_ranking")
trend_df = spark.read.parquet(f"{gold_path}/temperature_trend")
extremes_df = spark.read.parquet(f"{gold_path}/weather_extremes")

yearly_df.write.mode("overwrite").saveAsTable(
    "weather_catalog.weather_platform.yearly_city_summary"
)

monthly_df.write.mode("overwrite").saveAsTable(
    "weather_catalog.weather_platform.monthly_city_summary"
)

rainfall_df.write.mode("overwrite").saveAsTable(
    "weather_catalog.weather_platform.rainfall_ranking"
)

trend_df.write.mode("overwrite").saveAsTable(
    "weather_catalog.weather_platform.temperature_trend"
)

extremes_df.write.mode("overwrite").saveAsTable(
    "weather_catalog.weather_platform.weather_extremes"
)

print("Gold managed tables created successfully")

In [0]:
# Verify all 5 tables were created successfully
# Export complete: CSV files + tables ready for consumption
spark.sql("""
SHOW TABLES IN weather_catalog.weather_platform
""").show(truncate=False)